In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("../data/raw/UCI_Credit_Card.xlsx")
df.columns = df.columns.str.replace('# ', '', regex=False).str.strip()
df = df.drop(columns=["ID"])

In [4]:
#recreate model split
# Define target and features
y = df["default.payment.next.month"]
X = df.drop(columns=["default.payment.next.month"])

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
#recreate pipeline model
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=2000))
])

pipeline.fit(X_train, y_train)


Pipeline(steps=[('scaler', StandardScaler()),
                ('logreg', LogisticRegression(max_iter=2000))])

TASK 6- production batch simulation

In [6]:
# Simulate production batches
batch_size = 5000

batches = []

for i in range(0, len(X_test), batch_size):
    batches.append(X_test.iloc[i:i+batch_size])

len(batches)

2

TASK 7- Introduce controlled drift

In [7]:
#copy second batch
import numpy as np

baseline_batch = batches[0].copy()
drifted_batch = batches[1].copy()

In [9]:
#introduce drift
# Simulate higher credit limits
drifted_batch["LIMIT_BAL"] = drifted_batch["LIMIT_BAL"] * 1.4

# Simulate worsening payment behavior
drifted_batch["PAY_0"] = drifted_batch["PAY_0"] + 1

TASK 8- Calculate drift using PSI (population stability index)

In [10]:
#add PSI helper function
import numpy as np
import pandas as pd

def psi(expected, actual, buckets=10):
    """
    Population Stability Index (PSI)
    expected: baseline array/series
    actual: new array/series
    """
    expected = pd.Series(expected).dropna()
    actual = pd.Series(actual).dropna()

    # Create breakpoints using baseline (expected) quantiles
    breakpoints = np.percentile(expected, np.linspace(0, 100, buckets + 1))

    # Avoid duplicate bins (can happen if many same values)
    breakpoints = np.unique(breakpoints)
    if len(breakpoints) <= 2:
        return 0.0  # not enough variation to bin

    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts = np.histogram(actual, bins=breakpoints)[0]

    expected_perc = expected_counts / len(expected)
    actual_perc = actual_counts / len(actual)

    # Avoid division by zero
    expected_perc = np.where(expected_perc == 0, 1e-6, expected_perc)
    actual_perc = np.where(actual_perc == 0, 1e-6, actual_perc)

    psi_value = np.sum((actual_perc - expected_perc) * np.log(actual_perc / expected_perc))
    return psi_value

In [11]:
#compute PSI for the drifted features
psi_limit = psi(baseline_batch["LIMIT_BAL"], drifted_batch["LIMIT_BAL"])
psi_pay0  = psi(baseline_batch["PAY_0"], drifted_batch["PAY_0"])

print("PSI - LIMIT_BAL:", psi_limit)
print("PSI - PAY_0:", psi_pay0)

PSI - LIMIT_BAL: 0.48132682813756744
PSI - PAY_0: 5.161006566022142


TASK 9- Compute PSI for all features

In [12]:
psi_report = {}

for col in baseline_batch.columns:
    psi_value = psi(baseline_batch[col], drifted_batch[col])
    psi_report[col] = psi_value

psi_df = pd.DataFrame.from_dict(psi_report, orient="index", columns=["PSI"])
psi_df = psi_df.sort_values(by="PSI", ascending=False)

psi_df

,PSI
PAY_0,5.161007
LIMIT_BAL,0.481327
BILL_AMT6,0.014470
PAY_AMT5,0.012712
BILL_AMT5,0.010068
PAY_AMT1,0.008989
PAY_AMT4,0.008171
PAY_AMT6,0.007947
BILL_AMT2,0.005918
BILL_AMT4,0.005727


TASK 10- Build drift severity scoring engine

In [13]:
#categorize PSI
def categorize_psi(value):
    if value < 0.10:
        return "Stable"
    elif value < 0.25:
        return "Moderate Drift"
    else:
        return "Severe Drift"

psi_df["Drift_Level"] = psi_df["PSI"].apply(categorize_psi)

psi_df

,PSI,Drift_Level
PAY_0,5.161007,Severe Drift
LIMIT_BAL,0.481327,Severe Drift
BILL_AMT6,0.014470,Stable
PAY_AMT5,0.012712,Stable
BILL_AMT5,0.010068,Stable
PAY_AMT1,0.008989,Stable
PAY_AMT4,0.008171,Stable
PAY_AMT6,0.007947,Stable
BILL_AMT2,0.005918,Stable
BILL_AMT4,0.005727,Stable


In [14]:
#count severe drift features
severe_count = (psi_df["Drift_Level"] == "Severe Drift").sum()
moderate_count = (psi_df["Drift_Level"] == "Moderate Drift").sum()

print("Severe Drift Features:", severe_count)
print("Moderate Drift Features:", moderate_count)

Severe Drift Features: 2
Moderate Drift Features: 0


TASK 11- Overall batch risk score

In [15]:
#define batch risk logic
def compute_batch_risk(severe_count, moderate_count):
    if severe_count >= 2:
        return "High Risk"
    elif severe_count == 1 or moderate_count >= 2:
        return "Medium Risk"
    elif moderate_count == 1:
        return "Low Risk"
    else:
        return "Stable"

batch_risk = compute_batch_risk(severe_count, moderate_count)

print("Batch Risk Level:", batch_risk)

Batch Risk Level: High Risk


TASK 12- Prediction distribution drift

In [18]:
#compute prediction proba drift
# get prediction probabilities for both batches
baseline_proba = pipeline.predict_proba(baseline_batch)[:, 1]
drifted_proba = pipeline.predict_proba(drifted_batch)[:, 1]

psi_pred = psi(baseline_proba, drifted_proba)

print("PSI - Prediction Probabilities:", psi_pred)

PSI - Prediction Probabilities: 1.150008922862559


TASK 13- Performance drift monitoring

In [19]:
# Extract corresponding y values for batches
y_test_batches = []

for i in range(0, len(y_test), batch_size):
    y_test_batches.append(y_test.iloc[i:i+batch_size])

baseline_y = y_test_batches[0]
drifted_y = y_test_batches[1]

# Compute AUC for both batches
from sklearn.metrics import roc_auc_score

auc_baseline = roc_auc_score(baseline_y, baseline_proba[:len(baseline_y)])
auc_drifted = roc_auc_score(drifted_y, drifted_proba[:len(drifted_y)])

print("Baseline Batch AUC:", auc_baseline)
print("Drifted Batch AUC:", auc_drifted)

Baseline Batch AUC: 0.7100416888711254
Drifted Batch AUC: 0.6940068138053622
